# Sunglint over a tilted reference plane

The Cox-Munk statistics give the distribution of the slopes of the wave facets
**with respect to a reference plane**, the mean sea surface, usually taken
horizontal. In many situations the mean surface is not horizontal: the Earth
curvature seen from a low observer (see [the kumatage over a wind
gradient](kumatage_wind_gradient.ipynb)), the long waves of a swell carrying the
short wind waves (two-scale models), or a sloping water surface (river, weir,
wave tank).

![Sun-facet-sensor geometry over a horizontal and a tilted reference plane](../../../../illustration/research/tilted_reference_plane.png)

*Left*: horizontal reference plane, with the notation of the
{doc}`algorithm description <../../algorithms>`: the facet normal $\mathbf{n}$,
tilted by $\theta_n$, bisects the Sun and sensor directions. *Right*: the
reference plane is tilted by $\beta$, its normal $\mathbf{n}_p$ leaning toward
the azimuth $\phi_p$ (dashed: the horizontal plane). The Sun and viewing angles
$\theta_s'$ and $\theta_v'$ are now measured from $\mathbf{n}_p$.

When the reference plane is slanted by the tilt angle $\beta$, the Sun and
viewing angles, and therefore the facet slopes to input in the Cox-Munk
statistics, must be recomputed with respect to the new plane. This notebook
investigates four ways to do it, checks that the exact ones agree, and
quantifies the error of the approximate ones:

1. rotation of the Sun and viewing directions into the frame of the plane
   (implemented in {class}`coxmunk.coxmunk.sunglint` with `tilt` and `tilt_azi`),
2. closed-form spherical trigonometry for the local angles (vectorized),
3. rotation of the facet normal: the slopes are recomputed in the frame of the plane,
4. a small-tilt approximation: the plane slope is added to the facet slopes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from coxmunk import sunglint
from coxmunk.visu import plot

rad = np.radians

## Notation and the principal-plane case

The reference plane has the slope $\beta$ (tilt angle) and its normal leans
toward the azimuth $\phi_p$, counted from the Sun azimuth like the relative
azimuth $\phi$ of the sensor:

$$\mathbf{n}_p = (\sin\beta\cos\phi_p,\ \sin\beta\sin\phi_p,\ \cos\beta).$$

In the simplest case, the plane is tilted within the principal plane, its
normal leaning toward the Sun ($\phi_p = 0$), and the angles simply shift by
$\beta$:

$$\theta_s' = \theta_s - \beta, \qquad \theta_v' = \theta_v + \beta \quad (\phi = 180°),
\qquad \phi' = \phi.$$

The specular direction, $\theta_v' = \theta_s'$, is then at $\theta_v = \theta_s - 2\beta$:
tilting the mirror by $\beta$ moves the glint by $2\beta$.

In [ ]:
sza, ws = 40., 5.
vza = np.linspace(-80, 80, 321)        # negative: on the Sun side


def principal_plane(beta):
    I = []
    for v in vza:
        g = sunglint(sza, abs(v), 180. if v >= 0 else 0., tilt=beta, tilt_azi=0)
        I.append(g.sunglint(ws, 0, stats='bh2006')[0] * np.cos(g.sza) / np.cos(rad(sza)))
    return np.array(I)


fig, ax = plt.subplots(figsize=(10, 4.5))
for beta in (0, 5, 10, 20):
    line, = ax.plot(vza, principal_plane(beta), label=fr'$\beta$ = {beta}°')
    ax.axvline(sza - 2 * beta, color=line.get_color(), ls=':')
ax.set_xlabel('viewing zenith angle in the principal plane (deg, negative: Sun side)')
ax.set_ylabel(r'glint reflectance $\rho_h$')
ax.set_title(fr'Sun zenith angle {sza:.0f}°, plane tilted toward the Sun ($\phi_p = 0$), {ws:.0f} m/s')
ax.legend()
plt.show()

The reflectance $\rho_h$ is normalized by the irradiance on a horizontal
surface: {class}`~coxmunk.coxmunk.sunglint` returns a reflectance relative to the
irradiance on the tilted plane, $\rho'$, so that
$\rho_h = \rho' \cos\theta_s' / \cos\theta_s$. The dotted lines mark the
specular direction of the mean plane, $\theta_s - 2\beta$: the glint follows
it, its maximum being shifted away from the Sun by the $1/\mu_v'$ factor, as
for a horizontal surface.

## Method 1: rotation of the directions

The directions toward the Sun and toward the sensor,

$$\mathbf{s} = (\sin\theta_s, 0, \cos\theta_s), \qquad
\mathbf{v} = (\sin\theta_v\cos\phi, \sin\theta_v\sin\phi, \cos\theta_v),$$

are expressed in the frame of the plane with the rotation that brings the
vertical onto $\mathbf{n}_p$, a rotation by $\beta$ around the horizontal axis
perpendicular to $\phi_p$:

$$\mathbf{M} = R_z(\phi_p)\,R_y(\beta)\,R_z(-\phi_p), \qquad
\mathbf{s}' = \mathbf{M}^T\mathbf{s}, \quad \mathbf{v}' = \mathbf{M}^T\mathbf{v}.$$

The local angles $\theta_s'$, $\theta_v'$ and $\phi' = \phi_v' - \phi_s'$
follow, and the Cox-Munk computation is unchanged. This is what
{meth}`coxmunk.coxmunk.sunglint.local_geometry` does:

In [ ]:
g = sunglint(sza=40, vza=30, azi=120, tilt=15, tilt_azi=60)
print('local angles (deg): sza = {:.3f}, vza = {:.3f}, azi = {:.3f}'.format(*np.degrees([g.sza, g.vza, g.azi])))

## Method 2: closed-form spherical trigonometry

Writing the rotation explicitly gives, for a direction of zenith angle
$\theta$ and azimuth $\phi$,

$$\cos\theta' = \cos\beta\cos\theta + \sin\beta\sin\theta\cos(\phi - \phi_p),$$

$$\phi'_{abs} = \phi_p + \operatorname{atan2}\big(\sin\theta\sin(\phi - \phi_p),\
\cos\beta\sin\theta\cos(\phi - \phi_p) - \sin\beta\cos\theta\big),$$

applied to the Sun ($\phi = 0$) and to the sensor, with
$\phi' = \phi'_{abs,v} - \phi'_{abs,s}$. The first formula is the cosine
rule of the spherical triangle (normal of the plane, vertical, direction).
These formulas work on whole arrays of geometries:

In [ ]:
def local_angles(sza, vza, azi, beta, phip):
    """Local angles (deg) relative to the tilted plane, closed form, vectorized."""
    ts, tv, phi, b, p = map(rad, (sza, vza, azi, beta, phip))

    def rotate(theta, ph):
        mu = np.cos(b) * np.cos(theta) + np.sin(b) * np.sin(theta) * np.cos(ph - p)
        az = p + np.arctan2(np.sin(theta) * np.sin(ph - p),
                            np.cos(b) * np.sin(theta) * np.cos(ph - p) - np.sin(b) * np.cos(theta))
        return np.arccos(np.clip(mu, -1, 1)), az

    ts_l, fs_l = rotate(ts, np.zeros_like(phi))
    tv_l, fv_l = rotate(tv, phi)
    return np.degrees(ts_l), np.degrees(tv_l), np.degrees((fv_l - fs_l) % (2 * np.pi))


rng = np.random.default_rng(0)
geoms = rng.uniform([0, 0, 0, 0, 0], [70, 70, 360, 40, 360], (500, 5))   # sza, vza, azi, beta, phip
closed = np.array(local_angles(*geoms.T)).T
lib = []
for a in geoms:
    g = sunglint(*a[:3], tilt=a[3], tilt_azi=a[4])
    lib.append(np.degrees([g.sza, g.vza, g.azi]))
diff = np.abs(closed - np.array(lib))
diff[:, 2] = np.minimum(diff[:, 2], 360 - diff[:, 2])
diff[closed[:, 0] < 1e-3, 2] = 0     # Sun at the zenith of the plane: azimuth undefined
print('max difference with method 1 (deg):', diff.max())

## Method 3: rotation of the facet normal

Instead of the directions, the reflecting facet can be rotated. The facet
reflecting the Sun toward the sensor has the normal
$\mathbf{n} = (\mathbf{s} + \mathbf{v}) / |\mathbf{s} + \mathbf{v}|$,
independent of the reference plane. Its coordinates in the frame of the
plane, $\mathbf{n}' = \mathbf{M}^T\mathbf{n}$, give directly the inputs of the
Cox-Munk statistics:

$$z_x' = -\frac{n_x'}{n_z'}, \qquad z_y' = -\frac{n_y'}{n_z'}, \qquad
\cos\theta_n' = n_z',$$

rotated into the local Sun frame (by $-\phi_s'$) and then into the wind frame
(by $\phi_w$) to get $z_{up}$ and $z_{cr}$. The angle of incidence on the
facet, $\cos\omega = \mathbf{s}\cdot\mathbf{n}$, does not depend on the plane.
The glint is

$$\rho' = \frac{\pi\,P(z_{up}, z_{cr})\,R_{11}(\omega)}{4\,\mu_s'\,\mu_v'\,\cos^4\theta_n'},
\qquad \mu_s' = \mathbf{s}\cdot\mathbf{n}_p, \quad \mu_v' = \mathbf{v}\cdot\mathbf{n}_p.$$

This form makes explicit which quantities depend on the plane: the slopes
$z'$ and the cosines $\mu_s'$, $\mu_v'$, $\cos\theta_n'$; not the Fresnel
reflection. Below, it is written for arrays (statistics of
[BreonHenriot2006], shadowing omitted).

In [ ]:
def unit(theta, phi):
    theta, phi = np.broadcast_arrays(theta, phi)
    return np.stack([np.sin(theta) * np.cos(phi), np.sin(theta) * np.sin(phi), np.cos(theta)], -1)


def to_plane(x, beta, phip):
    """Coordinates of the vectors x (..., 3) in the frame of the plane: M^T x = Rz(phip) Ry(-beta) Rz(-phip) x."""
    b, p = rad(np.asarray(beta, float)), rad(np.asarray(phip, float))
    x1 = np.cos(p) * x[..., 0] + np.sin(p) * x[..., 1]
    y1 = -np.sin(p) * x[..., 0] + np.cos(p) * x[..., 1]
    x2 = np.cos(b) * x1 - np.sin(b) * x[..., 2]
    z2 = np.sin(b) * x1 + np.cos(b) * x[..., 2]
    return np.stack(np.broadcast_arrays(np.cos(p) * x2 - np.sin(p) * y1, np.sin(p) * x2 + np.cos(p) * y1, z2), -1)


def slope_pdf(zup, zcr, ws):
    """Gram-Charlier slope density of Breon and Henriot (2006), convention of Munk (2009)."""
    s_cr, s_up = np.sqrt(3e-3 + 1.85e-3 * ws), np.sqrt(1e-3 + 3.16e-3 * ws)
    c12, c30 = 9e-4 * ws ** 2, 0.45 / (1 + np.exp(7 - ws))
    c40, c22, c04 = 0.30, 0.12, 0.40
    xi, eta = zup / s_up, zcr / s_cr
    return np.exp(-(xi ** 2 + eta ** 2) / 2) / (2 * np.pi * s_cr * s_up) * (
        1 + c12 * xi * (1 - eta ** 2) / 2 - c30 * xi * (3 - xi ** 2) / 6
        + c40 * (3 - 6 * xi ** 2 + xi ** 4) / 24 + c22 * (1 - xi ** 2) * (1 - eta ** 2) / 4
        + c04 * (3 - 6 * eta ** 2 + eta ** 4) / 24)


def fresnel_r11(omega, m=1.334):
    root = np.sqrt(m ** 2 - np.sin(omega) ** 2)
    rl = (root - m ** 2 * np.cos(omega)) / (root + m ** 2 * np.cos(omega))
    rr = (np.cos(omega) - root) / (np.cos(omega) + root)
    return 0.5 * (rl ** 2 + rr ** 2)


def to_wind_frame(zx, zy, wazi):
    w = rad(wazi)
    return np.cos(w) * zx + np.sin(w) * zy, -np.sin(w) * zx + np.cos(w) * zy


def glint_facet_rotation(sza, vza, azi, beta, phip, ws, wazi=0):
    """Method 3: glint reflectance rho_h (horizontal irradiance) with the facet normal in the plane frame."""
    s, v = unit(rad(sza), 0 * rad(azi)), unit(rad(vza), rad(azi))
    n = (s + v) / np.linalg.norm(s + v, axis=-1, keepdims=True)
    s_l, v_l, n_l = (to_plane(x, beta, phip) for x in (s, v, n))
    mus, muv, mun = s_l[..., 2], v_l[..., 2], n_l[..., 2]
    zx, zy = -n_l[..., 0] / mun, -n_l[..., 1] / mun
    phs = np.arctan2(s_l[..., 1], s_l[..., 0])               # local Sun azimuth
    zx, zy = np.cos(phs) * zx + np.sin(phs) * zy, -np.sin(phs) * zx + np.cos(phs) * zy
    zup, zcr = to_wind_frame(zx, zy, wazi)
    omega = np.arccos(np.clip(np.sum(s * n, -1), -1, 1))
    rho_l = np.pi * slope_pdf(zup, zcr, ws) * fresnel_r11(omega) / (4 * mus * muv * mun ** 4)
    return np.where((mus > 0) & (muv > 0), rho_l * mus / np.cos(rad(sza)), 0.)


def glint_library(sza, vza, azi, beta, phip, ws, wazi=0):
    g = sunglint(sza, vza, azi, tilt=beta, tilt_azi=phip)
    if not g.visible:
        return 0.
    return g.sunglint(ws, wazi, stats='bh2006', shadow=False)[0] * np.cos(g.sza) / np.cos(rad(sza))


geoms = rng.uniform([5, 5, 0, 0, 0], [70, 70, 360, 30, 360], (300, 5))
ref = np.array([glint_library(*a, 7, 30) for a in geoms])
m3 = np.array([glint_facet_rotation(*a, 7, 30) for a in geoms])
ok = ref > 1e-3
print('max relative difference with method 1:', np.max(np.abs(m3[ok] / ref[ok] - 1)))

Methods 1, 2 and 3 are exact and equivalent: they differ only by what is
rotated (the directions or the facet) and by how the rotation is written.

## Method 4: small-tilt approximation

For a small tilt, the slope of a facet relative to the plane is approximately
the difference between its slope and the slope of the plane,
$\mathbf{z}_p = -\tan\beta\,(\cos\phi_p, \sin\phi_p)$:

$$z_x' \approx z_x + \tan\beta\cos\phi_p, \qquad z_y' \approx z_y + \tan\beta\sin\phi_p,$$

where $z_x$, $z_y$ are the slopes computed for a horizontal surface. This is
the classical way of tilting the short waves by the long waves in two-scale
models. Two variants are compared with the exact result:

* **4a**: the slopes are shifted, everything else is computed for a horizontal surface,
* **4b**: the slopes are shifted, and the cosines $\mu_v'$, $\cos\theta_n'$ are
  those of the tilted plane.

In [ ]:
def glint_slope_shift(sza, vza, azi, beta, phip, ws, wazi=0, local_cosines=False):
    """Method 4: horizontal slopes shifted by the plane slope; glint rho_h."""
    s, v = unit(rad(sza), 0 * rad(azi)), unit(rad(vza), rad(azi))
    n = (s + v) / np.linalg.norm(s + v, axis=-1, keepdims=True)
    zx, zy = -n[..., 0] / n[..., 2], -n[..., 1] / n[..., 2]
    zx = zx + np.tan(rad(beta)) * np.cos(rad(phip))
    zy = zy + np.tan(rad(beta)) * np.sin(rad(phip))
    zup, zcr = to_wind_frame(zx, zy, wazi)
    omega = np.arccos(np.clip(np.sum(s * n, -1), -1, 1))
    if local_cosines:
        muv, mun = to_plane(v, beta, phip)[..., 2], to_plane(n, beta, phip)[..., 2]
    else:
        muv, mun = v[..., 2], n[..., 2]
    return np.pi * slope_pdf(zup, zcr, ws) * fresnel_r11(omega) / (4 * s[..., 2] * muv * mun ** 4)


betas = np.array([0.5, 1, 2, 3, 5, 7.5, 10, 15, 20])
stats = {'4a': [], '4b': []}
for beta in betas:
    g = rng.uniform([5, 5, 0, 0, 0], [70, 70, 360, 0, 360], (2000, 5))
    g[:, 3] = beta
    exact = glint_facet_rotation(*g.T, 7)
    ok = exact > 1e-2                     # where the glint is not negligible
    for key, local in (('4a', False), ('4b', True)):
        err = np.abs(glint_slope_shift(*g.T, 7, local_cosines=local)[ok] / exact[ok] - 1)
        stats[key].append(np.percentile(100 * err, [50, 95]))

fig, ax = plt.subplots(figsize=(9, 4.5))
for (key, st), color in zip(stats.items(), ('C3', 'C0')):
    st = np.array(st)
    ax.plot(betas, st[:, 0], 'o-', color=color, label=f'{key}: median')
    ax.plot(betas, st[:, 1], 's--', color=color, label=f'{key}: 95th percentile')
ax.set_xlabel(r'tilt angle $\beta$ (deg)')
ax.set_ylabel('relative error on the glint (%)')
ax.set_title('Small-tilt approximation vs exact rotation (7 m/s, random geometries)')
ax.set_yscale('log')
ax.legend()
plt.show()

The median error grows about linearly with the tilt, roughly 1 % per degree:
with the horizontal cosines (4a), about 1 % at 1°, 6 % at 5° and 12 % at 10°.
Using the cosines of the tilted plane (4b) reduces it by about 30 %; the
remaining error comes from the linearization of the slope composition, which
dominates in the tails (the 95th percentiles of 4a and 4b are the same: about
4 % at 1° and 20 % at 5°). The approximation is suited to the small slopes of
long waves, not to large tilts.

The maps below compare the glint for a plane tilted by 10° across the
principal plane ($\phi_p = 90°$): the glint moves off the principal plane,
and the approximation reproduces the displacement but not the exact level.

In [ ]:
vza_g = np.linspace(0, 70, 36)
azi_g = np.linspace(0, 360, 91)
V, A = np.meshgrid(vza_g, azi_g)
beta, phip, sza_m = 10., 90., 30.

maps = {
    'horizontal plane': glint_facet_rotation(sza_m, V, A, 0., 0., 7),
    fr'exact, $\beta$ = {beta:.0f}°': glint_facet_rotation(sza_m, V, A, beta, phip, 7),
    'small-tilt approximation (4b)': glint_slope_shift(sza_m, V, A, beta, phip, 7, local_cosines=True),
}
fig, axs = plt.subplots(1, 3, subplot_kw=dict(projection='polar'), figsize=(18, 6))
p = plot()
for ax, (name, I) in zip(axs, maps.items()):
    p.add_polplot(ax, vza_g, azi_g, I.T, title=name, cmap=plt.cm.Spectral_r, vmin=0, vmax=0.6)
plt.tight_layout()

## Wind direction on a tilted plane

In {class}`~coxmunk.coxmunk.sunglint`, the wind azimuth `wazi` is counted
within the tilted plane from the local azimuth of the Sun. If the wind
direction is known in the horizontal frame (azimuth $\phi_w$ from the Sun), the
wind vector $\mathbf{w} = (\cos\phi_w, \sin\phi_w, 0)$ must be projected onto
the plane, $\mathbf{w}_p = \mathbf{w} - (\mathbf{w}\cdot\mathbf{n}_p)\,\mathbf{n}_p$,
and its azimuth taken in the frame of the plane:

In [ ]:
def local_wind_azimuth(wazi, sza, beta, phip):
    """Wind azimuth (deg) in the tilted plane, from the local Sun azimuth."""
    w = np.array([np.cos(rad(wazi)), np.sin(rad(wazi)), 0.])
    w_l = to_plane(w, beta, phip)              # the normal component vanishes in the plane frame
    s_l = to_plane(unit(rad(sza), 0.), beta, phip)
    return np.degrees(np.arctan2(w_l[1], w_l[0]) - np.arctan2(s_l[1], s_l[0])) % 360


beta, phip = 20., 90.
print(' wind azimuth (deg)   in the plane (deg)   glint change if ignored (%)')
for wazi in (0, 45, 90, 135):
    wl = local_wind_azimuth(wazi, 30., beta, phip)
    g = sunglint(30, 30, 150, tilt=beta, tilt_azi=phip)
    I_naive, I_proj = (g.sunglint(10, w, stats='bh2006')[0] for w in (wazi, wl))
    print(f'{wazi:12.0f} {wl:22.1f} {100 * (I_naive / I_proj - 1):24.1f}')

For a plane tilted by 20° across the principal plane, the wind direction seen
in the plane is shifted by about 30°, and ignoring the projection changes the
glint by 20 to 30 % for this 10 m/s wind: the wind direction must be projected
as soon as the slope anisotropy matters. A `wind_ref='global'` option could be
added to {class}`~coxmunk.coxmunk.sunglint` to do this projection internally.

## Polarization

The Stokes vector computed in the frame of the plane is referenced to the
local meridian plane (containing the normal of the plane and the viewing
direction). With `pol_ref='global'` (default),
{class}`~coxmunk.coxmunk.sunglint` rotates it back to the meridian plane of the
sensor containing the vertical, so that Q and U are comparable with those of a
horizontal surface. The degree of polarization does not depend on this choice.

## Summary

| Method | What is rotated | Exact | Vectorized here | Use |
|---|---|---|---|---|
| 1. Rotation of the directions | Sun and sensor | yes | no (library, scalar) | `sunglint(..., tilt, tilt_azi)` |
| 2. Spherical trigonometry | Sun and sensor | yes | yes | maps, large datasets |
| 3. Rotation of the facet normal | specular facet | yes | yes | explicit Cox-Munk inputs $z'_{up}$, $z'_{cr}$ |
| 4. Slope shift | — (linearized) | no | yes | small tilts (median error ~1 % per degree), two-scale models |

Exact methods require the reflectance normalization to be stated: relative to
the irradiance on the tilted plane ($\rho'$, output of the library) or on a
horizontal surface ($\rho_h = \rho'\,\mu_s'/\mu_s$, used in this notebook).